Pre comparation with 'train.csv' -> 'val.csv' (mantaining 'test.csv' intact for the final avaliation).

# 1. Dependencies import

In [ ]:
import sys
import os
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR != ROOT_DIR.parent and not (
    (ROOT_DIR / "data").exists() and (ROOT_DIR / "logs").exists()
):
    ROOT_DIR = ROOT_DIR.parent

os.chdir(ROOT_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.container import BarContainer
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    GradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
)
from xgboost import XGBClassifier

from logs.cmd.logger_config import setup_logger  # noqa: E402

## 1.2. Constants instances

In [ ]:
logger = setup_logger("model_benchmark", save_to_file=True)

SEED = 42
DATA_DIR = ROOT_DIR / "data" / "refined"

images_dir = ROOT_DIR / "tests" / "results" / "images"
images_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

TARGET_COL = "target_risk"

COLOR_PURPLE_DARK = "#4E31A4"
COLOR_PURPLE_LIGHT = "#D6CFF8"
COLOR_YELLOW_SECONDARY = "#D97706"

# 2. Data loading

In [ ]:
train_norm = pd.read_csv(DATA_DIR / "train.csv")
val_norm = pd.read_csv(DATA_DIR / "val.csv")

sens_train = pd.read_csv(DATA_DIR / "sensitive_train.csv")
sens_val = pd.read_csv(DATA_DIR / "sensitive_val.csv")

X_train_norm = train_norm.drop(columns=[TARGET_COL])
y_train = train_norm[TARGET_COL].astype(int)

X_val_norm = val_norm.drop(columns=[TARGET_COL])
y_val = val_norm[TARGET_COL].astype(int)

# codify only race1 and gender (fam_inc and age are already standardized in X_train_norm)
ohe_sens = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
sens_cols_to_add = ["race1", "gender"]

sens_train_encoded = pd.DataFrame(
    np.asarray(ohe_sens.fit_transform(sens_train[sens_cols_to_add])),
    columns=ohe_sens.get_feature_names_out(sens_cols_to_add),
    index=X_train_norm.index,
)
sens_val_encoded = pd.DataFrame(
    np.asarray(ohe_sens.transform(sens_val[sens_cols_to_add])),
    columns=ohe_sens.get_feature_names_out(sens_cols_to_add),
    index=X_val_norm.index,
)

X_train_sens = pd.concat([X_train_norm, sens_train_encoded], axis=1)
X_val_sens = pd.concat([X_val_norm, sens_val_encoded], axis=1)

datasets = {
    "1. Normal (Without sensible features)": (X_train_norm, X_val_norm),
    "2. With sensible features (+ race1, gender)": (X_train_sens, X_val_sens),
}

logger.info(f"Normal Features: {X_train_norm.shape[1]} | Features With Sensible Features: {X_train_sens.shape[1]}")
logger.info(f"Risk Prevalence (y=1) -> Training: {y_train.mean():.2%} | Validation: {y_val.mean():.2%}")

# 3. Possible models definition and Metrics + Fairness

In [ ]:
def build_models(seed: int = SEED, pos_weight: float = 1.0) -> dict:
    return {
        "Baseline": DummyClassifier(strategy="most_frequent", random_state=seed),
        "Logistic Regression (L2)": LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=1000,
            random_state=seed,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            n_jobs=-1,
            random_state=seed,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=200,
            max_depth=5,
            learning_rate=0.05,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=seed,
        ),
        "MLP (Rede Neural)": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.01,
            max_iter=300,
            early_stopping=True,
            random_state=seed,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pos_weight,
            eval_metric="logloss",
            random_state=seed,
        )
    }


def compute_fairness_gaps(y_true: pd.Series, y_pred: np.ndarray, race_series: pd.Series) -> dict:
    """
    Calculate fairness metrics between racial groups in the validation set:
    - Statistical Parity Difference (Selection Rate Black - White)
    - Equal Opportunity Gap (Recall Black - Recall White)
    """
    y_true_arr = y_true.to_numpy()
    mask_white = (race_series == "white").to_numpy()
    mask_black = (race_series == "black").to_numpy()

    sel_white = float(y_pred[mask_white].mean())
    sel_black = float(y_pred[mask_black].mean())

    tpr_white = float(recall_score(y_true_arr[mask_white], y_pred[mask_white], zero_division=0))
    tpr_black = float(recall_score(y_true_arr[mask_black], y_pred[mask_black], zero_division=0))

    return {
        "SPD_Gap (Black - White)": round(sel_black - sel_white, 3),
        "TPR_Gap (Black - White)": round(tpr_black - tpr_white, 3),
    }

# 4. Experiment run

In [ ]:
results = []
neg_pos_ratio = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

for dataset_name, (X_tr, X_va) in datasets.items():
    models = build_models(SEED, pos_weight=neg_pos_ratio)

    for model_name, model in models.items():
        model.fit(X_tr, y_train)

        # positive class probabilities
        if hasattr(model, "predict_proba"):
            y_prob = np.asarray(model.predict_proba(X_va))[:, 1]
        else:
            y_prob = np.asarray(model.predict(X_va), dtype=float)

        # for models without native class_weight, adjust threshold closer to prevalence (~0.15)
        needs_custom_thresh = ("MLP" in model_name) or (model_name == "GradientBoosting")
        threshold = 0.15 if needs_custom_thresh else 0.50
        y_pred = (y_prob >= threshold).astype(int)

        fairness = compute_fairness_gaps(y_val, y_pred, sens_val["race1"])

        roc_auc = float(roc_auc_score(y_val, y_prob))
        pr_auc = float(average_precision_score(y_val, y_prob))
        f1 = float(f1_score(y_val, y_pred, zero_division=0))
        rec = float(recall_score(y_val, y_pred, zero_division=0))
        prec = float(precision_score(y_val, y_pred, zero_division=0))

        results.append(
            {
                "Scenario": dataset_name,
                "Model": model_name,
                "ROC-AUC": round(roc_auc, 4),
                "PR-AUC": round(pr_auc, 4),
                "F1 (Risk=1)": round(f1, 4),
                "Recall (Risk=1)": round(rec, 4),
                "Precision (Risk=1)": round(prec, 4),
                **fairness,
            }
        )

results_df = pd.DataFrame(results)
results_df.sort_values(by=["ROC-AUC", "F1 (Risk=1)"], ascending=False)

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=["ROC-AUC", "F1 (Risk=1)"], ascending=False
)

results_dir = ROOT_DIR / "tests" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "model_benchmark_results.csv"
results_df.to_csv(csv_path, index=False)
logger.info(f"Results saved at: {csv_path}")

results_df

# 5. Visualization

In [ ]:
METRIC_SLUGS = {
    "ROC-AUC": "roc_auc",
    "PR-AUC": "pr_auc",
    "F1 (Risk=1)": "f1_risk_1",
    "Recall (Risk=1)": "recall_risk_1",
    "SPD_Gap (Black - White)": "fairness_spd_gap",
    "TPR_Gap (Black - White)": "fairness_tpr_gap",
}

metrics = list(METRIC_SLUGS.keys())
filtered_df = results_df[results_df["Model"] != "Baseline"]
palette = [COLOR_PURPLE_DARK, COLOR_YELLOW_SECONDARY]

for metric in metrics:
    fig, ax = plt.subplots(figsize=(8.5, 5.0))

    sns.barplot(
        data=filtered_df,
        x="Model",
        y=metric,
        hue="Scenario",
        palette=palette,
        edgecolor=COLOR_PURPLE_DARK,
        linewidth=0.8,
        ax=ax,
    )

    for container in ax.containers:
        if isinstance(container, BarContainer):
            ax.bar_label(container, fmt="%.3f", fontsize=8.5, padding=3)

    ax.set_title(
        f"Comparação por Modelo — {metric}",
        fontsize=12,
        fontweight="bold",
        color=COLOR_PURPLE_DARK,
        pad=58,
    )
    ax.set_xlabel("")
    ax.set_ylabel(metric, fontsize=10, fontweight="bold")

    min_val = float(filtered_df[metric].min())
    max_val = float(filtered_df[metric].max())

    if metric in {"SPD_Gap (Black - White)", "TPR_Gap (Black - White)"}:
        y_low = min(0.0, min_val * 1.25)
        y_high = max(0.05, max_val * 1.25)
        ax.set_ylim(y_low, y_high)
        ax.axhline(
            0.0,
            color=COLOR_PURPLE_DARK,
            linestyle=":",
            linewidth=1.0,
            label="_nolegend_",
        )
    else:
        ax.set_ylim(0, min(1.05, max_val * 1.14))

    ax.grid(axis="y", linestyle="--", alpha=0.9, color=COLOR_PURPLE_LIGHT)
    ax.set_axisbelow(True)

    plt.xticks(rotation=15, ha="right")
    ax.legend(
        title="Cenário",
        frameon=True,
        facecolor="white",
        edgecolor=COLOR_PURPLE_LIGHT,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
    )
    sns.despine(top=True, right=True)
    plt.tight_layout()

    file_path = images_dir / f"{METRIC_SLUGS[metric]}_{timestamp}.png"
    fig.savefig(file_path, dpi=300, bbox_inches="tight")
    logger.info(f"Chart saved at salvo em: {file_path}")

    plt.show()